In [ ]:
import asyncio
import json
import logging
import copy
from typing import Any, Dict, List, Optional
from dataclasses import dataclass
import torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
import time
import nest_asyncio
from concurrent.futures import ThreadPoolExecutor

# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Configure logging for notebook
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("✅ Imports successful!")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
print(f"🔥 Number of GPUs: {torch.cuda.device_count()}")

In [ ]:
import sys
sys.path.append('/home/sagemaker-user/csbai/multiturn_rl')
for _ in sys.path:
    print(_)

from simulators.conversation_simulator import ConversationConfig, MultiTurnConversationGenerator
print("✅ Conversation Generator class defined")

In [ ]:
test_config = ConversationConfig(
    assistant_meta_prompt="You are a helpful cooking assistant. Provide clear, step-by-step cooking instructions and tips.",
    user_meta_prompt="You are a user asking an assistant about the following things. Generate response based on the following conversation: \n {chat_history}",
    max_total_turns=8,
    max_gen_workers=2,
    local_model_path="/home/sagemaker-user/csbai/multiturn_rl/outputs/sft/inspired/test",  # Your LoRA adapter path
    base_model_path="meta-llama/Llama-3.2-1B-Instruct",  # Your base model path
    assistant_generation_kwargs={
        "temperature": 0.7,
        "max_tokens": 256
    },
    user_generation_kwargs={
        "model": "anthropic.claude-3-sonnet-20240229-v1:0",  # or your preferred bedrock model
        "temperature": 1.0,
        "max_tokens": 512,
        "num_retries": 10  # For UserSimulator's retry logic
    },
    batch_size=5,
    enable_batching=True
)

print("✅ Test configuration created")
print(f"📝 Task: {test_config.user_meta_prompt}")
print(f"🔄 Max turns: {test_config.max_total_turns}")
print(f"⚡ Max workers: {test_config.max_gen_workers}")
print(f"📦 Batch size: {test_config.batch_size}")  # CHANGED: Added batch info

In [ ]:
async def test_single_conversation():
    """Test generating a single conversation"""
    print("🧪 Testing single conversation generation...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    test_prompt = "How do I make chocolate chip cookies?"
    
    conversation = await generator.generate_single_conversation(test_prompt)
    
    if conversation:
        print("\n" + "="*50)
        print("📋 GENERATED CONVERSATION:")
        print("="*50)
        
        for i, msg in enumerate(conversation):
            role_emoji = {"system": "⚙️", "user": "👤", "assistant": "🤖"}
            print(f"\n{role_emoji.get(msg['role'], '❓')} {msg['role'].upper()}:")
            print(f"   {msg['content']}")
        
        print("\n" + "="*50)
        print(f"✅ Success! Generated {len(conversation)} messages")
        return conversation
    else:
        print("❌ Failed to generate conversation")
        return None

# Run the test
print("🚀 Starting single conversation test...")
single_conv_result = await test_single_conversation()

In [ ]:
single_conv_result

In [ ]:
async def test_batch_conversations():
    """Test generating multiple conversations in parallel - ENHANCED"""
    print("🧪 Testing batch conversation generation...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    # CHANGED: Start with fewer prompts for debugging
    test_prompts = [
        "How do I make chocolate chip cookies?",
        "What's the best way to cook pasta?",
        "How can I make a simple salad?"
    ] * 20
    
    print(f"📝 Testing with {len(test_prompts)} prompts...")
    
    # CHANGED: Use new batch generation method with debugging
    start_time = time.time()
    
    print("🔍 Starting batch generation...")
    results = await generator.generate_conversations_batch(test_prompts, len(test_prompts))
    
    end_time = time.time()
    
    # Filter out None results
    successful_conversations = [conv for conv in results if conv is not None]
    
    print(f"\n⏱️  Total time: {end_time - start_time:.2f} seconds")
    print(f"✅ Successfully generated {len(successful_conversations)}/{len(test_prompts)} conversations")
    
    # Show summary
    for i, conv in enumerate(successful_conversations):
        if conv:
            print(f"   Conversation {i+1}: {len(conv)} messages")
            if i < len(test_prompts):
                print(f"      Prompt: {test_prompts[i][:50]}...")
    
    return successful_conversations

# ADDED: Simple sequential test for comparison
async def test_sequential_conversations():
    """Test generating conversations one by one for debugging"""
    print("🧪 Testing SEQUENTIAL conversation generation for debugging...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    test_prompts = [
        "How do I make chocolate chip cookies?",
        "What's the best way to cook pasta?"
    ]
    
    results = []
    for i, prompt in enumerate(test_prompts):
        print(f"\n🔄 Starting conversation {i+1}/{len(test_prompts)}")
        start_time = time.time()
        
        result = await generator.generate_single_conversation(prompt)
        
        end_time = time.time()
        print(f"⏱️  Conversation {i+1} took {end_time - start_time:.2f} seconds")
        
        if result:
            print(f"✅ Conversation {i+1} completed with {len(result)} messages")
            results.append(result)
        else:
            print(f"❌ Conversation {i+1} failed")
            results.append(None)
    
    return results

# ADDED: Simple batch test without the complex batching logic
async def test_simple_batch():
    """Test simple concurrent execution without complex batching"""
    print("🧪 Testing SIMPLE batch conversation generation...")
    
    generator = MultiTurnConversationGenerator(test_config)
    
    test_prompts = [
        "How do I make chocolate chip cookies?",
        "What's the best way to cook pasta?",
        "How can I make a simple salad?"
    ] 
    
    print(f"📝 Testing with {len(test_prompts)} prompts...")
    
    # SIMPLE: Just create tasks and wait for them
    start_time = time.time()
    
    # Create all tasks
    tasks = []
    for i, prompt in enumerate(test_prompts):
        print(f"📝 Creating task {i+1}: {prompt[:30]}...")
        task = asyncio.create_task(generator.generate_single_conversation(prompt))
        tasks.append(task)
    
    print(f"⏳ Waiting for {len(tasks)} tasks...")
    
    # Wait for all tasks with timeout
    try:
        results = await asyncio.wait_for(asyncio.gather(*tasks), timeout=300)  # 5 minute timeout
        print("✅ All tasks completed!")
    except asyncio.TimeoutError:
        print("❌ Tasks timed out after 5 minutes")
        return []
    
    end_time = time.time()
    
    # Filter successful results
    successful = [r for r in results if r is not None]
    
    print(f"\n⏱️  Total time: {end_time - start_time:.2f} seconds")
    print(f"✅ Successfully generated {len(successful)}/{len(test_prompts)} conversations")
    
    return successful

# # Run the simple batch test first
# print("🚀 Starting SIMPLE batch test...")
# simple_batch_results = await test_simple_batch()

In [ ]:
print("\n" + "="*50)
print("🚀 Now testing the complex batch method...")
batch_results = await test_batch_conversations()

In [ ]:
# 🔴 NEW TEST: Test early termination handling
async def test_early_termination():
    """Test conversations with different termination points"""
    print("🧪 Testing early termination handling...")
    
    # Create a custom user simulator that terminates at different points
    class VariableTerminationUserSimulator:
        def __init__(self, termination_turn, task_desc='', single_turn_prompt='', **kwargs):
            self.termination_turn = termination_turn
            self.call_count = 0
            
        def __call__(self, messages: List[dict]) -> str:
            self.call_count += 1
            
            if self.call_count >= self.termination_turn:
                return "Perfect! Thank you for your help!"
            else:
                return f"That's helpful. Can you tell me more about step {self.call_count + 1}?"
    
    # Temporarily modify the generator to use our custom simulator
    generator = MultiTurnConversationGenerator(test_config)
    
    # Create test scenarios with different termination points
    test_scenarios = [
        ("Quick pasta recipe?", 2),  # Terminates after 2 turns
        ("How to make a complex French dish?", 5),  # Terminates after 5 turns
        ("Simple sandwich instructions?", 1),  # Terminates after 1 turn
        ("Detailed cake baking process?", 4),  # Terminates after 4 turns
        ("Quick salad?", 3),  # Terminates after 3 turns
    ]
    
    # Create custom conversation states
    conversation_states = []
    for i, (prompt, term_turn) in enumerate(test_scenarios):
        state = {
            'id': i,
            'prompt': prompt,
            'chat_history': [
                {"role": "system", "content": test_config.task_desc},
                {"role": "user", "content": prompt}
            ],
            'user_sim': VariableTerminationUserSimulator(
                termination_turn=term_turn,
                task_desc=test_config.task_desc,
                single_turn_prompt=prompt
            ),
            'completed': False,
            'turn_count': 0
        }
        conversation_states.append(state)
    
    print(f"📝 Testing {len(test_scenarios)} conversations with different termination points...")
    print("Expected terminations:", [t[1] for t in test_scenarios])
    
    start_time = time.time()
    
    # Process conversations in rounds
    max_rounds = test_config.max_total_turns // 2
    
    for round_idx in range(max_rounds):
        # Filter active conversations
        active_states = [s for s in conversation_states if not s['completed']]
        
        if not active_states:
            print(f"✅ All conversations completed by round {round_idx}")
            break
        
        print(f"🔄 Round {round_idx + 1}: Processing {len(active_states)} active conversations")
        
        # Generate assistant responses in batch
        await generator._process_assistant_turn_batch(active_states)
        
        # Check for terminations and generate user responses
        await generator._process_user_turn_batch(active_states)
        
        # Update turn counts
        for state in active_states:
            state['turn_count'] += 1
    
    end_time = time.time()
    
    # Show results
    print(f"\n⏱️  Total time: {end_time - start_time:.2f} seconds")
    print("\n📊 Results:")
    for i, (state, (prompt, expected_term)) in enumerate(zip(conversation_states, test_scenarios)):
        actual_turns = state['turn_count']
        print(f"   Conv {i+1}: Expected {expected_term} turns, got {actual_turns} turns - {prompt[:30]}...")
        print(f"            Messages in conversation: {len(state['chat_history'])}")
    
    return [state['chat_history'] for state in conversation_states]

# Run early termination test
early_term_results = await test_early_termination()